# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități masive de text, capabil să înțeleagă, să genereze și să interacționeze cu limbajul uman într-un mod fluent și coerent. Datorită acestui antrenament extins, ele pot executa o gamă largă de sarcini lingvistice, de la traducere și rezumare la scrierea creativă și răspunsuri la întrebări complexe.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text pentru a înțelege, genera și interacționa în limbaj natural. Această capacitate îi permite să efectueze o varietate de sarcini, de la scrierea de texte creative și rezumate, la răspunsuri la întrebări complexe și traduceri.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 6 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 75 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, România a avut multiple schimbări de guvern. Partidele politice au format și dizolvat alianțe frecvent. Au existat tensiuni legate de independența justiției. S-au implementat reforme în sistemul de sănătate și educație. Au apărut noi formațiuni politice pe scena publică. Alegerile au reconfigurat constant peisajul politic.

--- Gemini 2.5 Flash ---
Iată un rezumat în 6 propoziții scurte:

Politica românească a fost marcată de o instabilitate guvernamentală inițială. S-au succedat guverne minoritare și coaliții diverse, inclusiv PNL-USR PLUS-UDMR. Ulterior, s-a format o coaliție majoritară inedită între PNL, PSD și UDMR. Alegerile parlamentare din 2020 au adus o fragmentare crescută și apariția unor noi partide. Agenda politică a fost puternic influențată de pandemia COVID-19 și de crizele internaționale. S-au înregistrat modificări legislative în domeniul justiției și un accent pe implementarea PNRR.

--- OpenRouter Free ---
1. În 2019, 

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un asistent de cercetare foarte bun care adnotează comentarii politice.
Răspunzi scurt, in maxim 3 propozitii scurte, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli intotdeauna plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Cinic și frustrat.
Emoție dominantă: Furie și neîncredere.
Țintă principală: Clasa politică.
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, resemnat și frustrat.
Emoție dominantă: Frustrare, deziluzie și furie.
Țintă principală: Clasa politică în ansamblul ei și sistemul democratic.
Populism: da

--- OpenRouter Free ---
Ton: critic și dezamăgit.  
Emoție dominantă: indignare și neputință.  
Țintă principală: clasa politică în ansamblu.  
Populism: da.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Toți politicienii fură mult, iar oamenii simpli plătesc mereu toata nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare de elita care adnotează comentarii politice bune."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă furie față de politicieni, acuzându-i de corupție și ignorarea poporului, sugerând o dihotomie între "ei" (politicienii) și "noi" (oamenii simpli).'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă furie și dezamăgire față de politicieni, acuzându-i de corupție și de ignorarea poporului, folosind o retorică populistă care opune 'oamenii simpli' elitei politice."}

--- OpenRouter Free ---
[Eroare: JSONDecodeError — Expecting value: line 4803 column 1 (char 26411)]


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 4 propoziții ce poate însemna acest lucru pentru viața politică si publica.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.5, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi scrutinuri, ceea ce implică un proces electoral repetat. Această situație poate genera incertitudine politică și poate influența stabilitatea guvernamentală. De asemenea, poate afecta încrederea publicului în procesul electoral și în instituțiile democratice. În consecință, poate fi necesară o reevaluare a procedurilor electorale pentru a preveni astfel de situații în viitor.

temperature=0.5:
Anularea alegerilor de către Curtea Constituțională poate conduce la organizarea de noi scrutinuri, generând o perioadă de incertitudine politică și posibile reconfigurări ale forțelor politice. Această decizie poate influența legitimitatea instituțiilor alese și poate necesita eforturi suplimentare pentru restabilirea încrederii publice în procesul electoral. De asemenea, poate duce la o perioadă de instab

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da  | da  | da | nu | stabil, bun|
| Gemini 2.5 Flash  | da  | da  | da | nu |  bun|
| OpenRouter Free | parțial | da | parțial | da | erori JSON, instabil|

### Decizie
**Model principal ales:**  Gemini 2.5 Flash Lite
**Model de rezervă:**  OpenRouter Free
**Temperature recomandată:**  0.1
**De ce am ales acest model?**  Modelul Gemini 2.5 Flash Lite ofera raspunsuri consistente, corecte si stabile.
Respecta instructiunile si genereaza JSON valid, fiind potrivit pentru adnotare.
OpenRouter este ales ca model de rezerva deoarece ofera o alternativa disponibila in cazul in care modelul principal nu poate fi utilizat (de exemplu, din cauza limitelor de API). Desi prezinta unele erori si instabilitate, poate furniza raspunsuri utile in situatii de fallback.


## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.1

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales